# 권리기준일별 배당 이력과 지급일

SEIBRO Open API에서 회사의 배당일정과 배당분배금내역을 권리기준일 단위로 조회합니다.
분기 배당도 포함되며 보통주와 우선주는 표준코드로 구별합니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/seibro-api/blob/main/scenarios/openapi/03_dividend_history.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `SEIBRO_API_KEY`를 등록하고 이 노트북의 액세스를 켭니다. SEIBRO Open API 키는 [SEIBRO OpenPlatform](https://seibro.or.kr/OpenPlatform/)에서 신청합니다. 아래 두 셀을 실행해 패키지를 설치하고 실행 환경을 준비하세요.

In [1]:
# Colab에서만: 패키지 설치 (로컬은 이 레포를 pip install -e . 로 설치해 두면 됨)
import sys
if "google.colab" in sys.modules:
    !pip -q install "git+https://github.com/beaten-by-the-market/seibro-api.git" koreanize-matplotlib

In [2]:
SECRETS = ['SEIBRO_API_KEY']
# ── SEIBRO 공통 셀 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 아래 SECRETS 이름으로 값을 넣고 "노트북 액세스"를 켜세요.
#        SEIBRO_API_KEY = 예탁결제원 SEIBRO Open API 인증키 (https://seibro.or.kr/OpenPlatform)
#        DART_API_KEY = OpenDART 인증키 (DART 데이터를 쓰는 노트북만)
#        KRX_ID / KRX_PW = KRX 정보데이터시스템 계정 (KRX 데이터를 쓰는 노트북만)
# 로컬: 환경변수 또는 작업 폴더(레포 루트)의 .env 를 읽습니다.
# 키·비밀번호 값은 어디에도 출력하지 않습니다. 있는지 여부만 보여 줍니다.
import os, sys, warnings
from datetime import date, datetime, timedelta

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

if IN_COLAB and SECRETS:
    from google.colab import userdata
    for _name in SECRETS:  # noqa: F821  (빌더가 이 셀 맨 위에 SECRETS 목록을 넣음)
        if os.environ.get(_name):
            continue
        try:
            _v = userdata.get(_name)
        except Exception:
            _v = None
        if not _v:
            from getpass import getpass
            _v = getpass(f"{_name}: ")
        os.environ[_name] = _v.strip()
elif SECRETS:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))

_missing = [
    n for n in SECRETS  # noqa: F821
    if not os.environ.get(n) and (IN_COLAB or not n.startswith("KRX_"))
]
if _missing:
    raise RuntimeError(f"{_missing} 가 없습니다. Colab Secrets 또는 .env 에 넣어 주세요.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def won(v, digits=1):
    """원 → '1.2조' / '5,027억' / '3,400만'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


def shares(v):
    """주식수 → '23,607,980주'."""
    return "-" if v is None or pd.isna(v) else f"{int(v):,}주"


# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

def _status(n):
    if os.environ.get(n):
        return "있음"
    # 로컬에서 krx-data-api를 편집 설치(pip install -e)했다면 그 레포의 .env를 패키지가 직접 읽는다
    return "없음" if IN_COLAB or not n.startswith("KRX_") else "krx-data-api 자체 .env 사용"


credential_status = ", ".join(f"{n}: {_status(n)}" for n in SECRETS)
print("SEIBRO 준비 완료", "(Colab)" if IN_COLAB else "(로컬)",
      "| " + (credential_status or "웹 조회 · 키 필요 없음"))

SEIBRO 준비 완료 (로컬) | SEIBRO_API_KEY: 있음


In [3]:
from IPython.display import display

from seibro_api import get_dividend_history

STOCK_CODE = "005930"  # 삼성전자
START_DT = "20240101"
END_DT = "20251231"

In [4]:
df = get_dividend_history(STOCK_CODE, START_DT, END_DT, save_csv=False)
if df.empty:
    print("해당 기간에 조회된 배당 이력이 없습니다.")
else:
    display(df.filter(items=[
        "회사명", "표준코드", "종목명", "주식종류", "권리기준일", "배당구분", "확정구분",
        "주당배당금", "시가배당률(%)", "현금배당지급일",
    ]))


[Seibro] 배당 이력 조회 (Open API)
  -> 삼성전자 (005930, 고객번호: 593)
  -> 기간 20240101~20251231, 1개 구간으로 분할 호출


  [배당일정] 20240101~20251231: 8건


  [배당분배금] 20240101~20251231: 16건
  -> 총 16행 (20240331~20251231)


,회사명,표준코드,종목명,주식종류,권리기준일,배당구분,확정구분,주당배당금,시가배당률(%),현금배당지급일
0,삼성전자,KR7005930003,삼성전자,보통주,20251231,현금배당,확정,566,0.50,20260417
1,삼성전자,KR7005931001,삼성전자1우,우선주,20251231,현금배당,확정,567,0.70,20260417
2,삼성전자,KR7005930003,삼성전자,보통주,20250930,현금배당,확정,370,0.40,20251119
3,삼성전자,KR7005931001,삼성전자1우,우선주,20250930,현금배당,확정,370,0.60,20251119
4,삼성전자,KR7005930003,삼성전자,보통주,20250630,현금배당,확정,367,0.60,20250820
5,삼성전자,KR7005931001,삼성전자1우,우선주,20250630,현금배당,확정,367,0.70,20250820
6,삼성전자,KR7005930003,삼성전자,보통주,20250331,현금배당,확정,365,0.60,20250520
7,삼성전자,KR7005931001,삼성전자1우,우선주,20250331,현금배당,확정,365,0.70,20250520
8,삼성전자,KR7005930003,삼성전자,보통주,20241231,현금배당,확정,363,0.70,20250418
9,삼성전자,KR7005931001,삼성전자1우,우선주,20241231,현금배당,확정,364,0.80,20250418


배당 기준일이 범위 안에 있으면 지급일이 범위 밖일 수 있습니다.
결산연도 기준 배당성향은 [웹 배당내역상세](../web/01_dividend_details.md)에 있습니다.